In [0]:
# Databricks notebook source

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, IntegerType, DateType, BooleanType, StringType
from pyspark.sql.window import Window

def transform_customer(customer_df):

    windowSpec_rw = Window.partitionBy("CustomerID").orderBy("CustomerID")

    customer_df = customer_df.withColumn("TerritoryID",F.when(F.col("TerritoryID").isNull(), 0).otherwise(F.col("TerritoryID"))).withColumn(
		"rw", F.row_number().over(windowSpec_rw))
    customer_df = customer_df.filter(F.col("rw") == 1).drop("rw")
    customer_df = customer_df.withColumn("processed_timestamp", F.current_timestamp())

    customer_df = customer_df.select(    
      F.col("CustomerID").cast(IntegerType()).alias("CustomerID"),
      F.col("PersonID").cast(IntegerType()).alias("PersonID"),
      F.col("StoreID").cast(IntegerType()).alias("StoreID"),
      F.col("TerritoryID").cast(IntegerType()).alias("TerritoryID"),
      F.trim(F.col("AccountNumber")).alias("AccountNumber"),
      F.col("rowguid").alias("rowguid"),
      F.col("ModifiedDate").cast(DateType()).alias("ModifiedDate"),
      F.col("_rescued_data").alias("_rescued_data"),
      F.col("processed_timestamp").alias("processed_timestamp")
    )
                                 
    return customer_df




if __name__ == "__main__":

    customer_tbl = dbutils.widgets.get("customer")
    customer_df = df = spark.read.table(customer_tbl)
    customer_df_tgt = transform_customer(customer_df)
    display(customer_df_tgt)